# Phase 5C — Phân tích tuning có thể tái lập

Chạy tại máy cá nhân. ZIP gốc là nguồn thực nghiệm duy nhất; notebook chỉ kiểm tra và phân tích CPU, không chạy tuning, training hoặc evaluation mô hình. Các điểm số dưới đây là **validation**, không phải final test.

Cài trước: `python -m pip install -r requirements-analysis.txt`. Đặt ZIP gốc tại `artifacts/phase5c/` trong repository, hoặc override `SEQREC_PHASE5C_ARCHIVE`. Không có đường dẫn Downloads hard-code. Checkpoint công khai được tải riêng bằng `python scripts/fetch_phase5c_checkpoints.py`; gói checkpoint không thay thế ZIP gốc khi audit toàn bộ 15 trials. Notebook đã thực thi được lưu riêng dưới `results/derived/`.


In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "src/analysis/phase5c.py").is_file(), "Open from repository root or notebooks/"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "scripts"))
from src.analysis.phase5c import analyze_archive, ARCHIVE_SHA256, resolve_archive_path
ARCHIVE = resolve_archive_path(ROOT)
assert ARCHIVE.is_file(), f"Place original private ZIP here or set SEQREC_PHASE5C_ARCHIVE: {ARCHIVE}"
print("Expected source SHA256:", ARCHIVE_SHA256)


## 1. Xác minh tính toàn vẹn và provenance

Kiểm tra CRC, toàn bộ member hashes, 150 status-hashed artifacts, 30 best/latest checkpoints, plan/config fingerprints, validation population, contributions, top-K, selection và stopping counters. Raw extraction không ghi đè tệp khác hash.

Dataset bytes không nằm trong ZIP: chỉ xác minh tính nhất quán của split IDs và báo cáo data audit được lưu. ZIP không có Git/source attestation của phiên training.


In [ ]:
trials, audit = analyze_archive(ARCHIVE, ROOT)
assert audit["state"] == "VERIFIED" and audit["trials"] == 15
assert audit["test_result_artifacts"] == 0
from IPython.display import display, Image, Markdown
print({k: audit[k] for k in ("archive_files", "trials", "checkpoint_files_verified", "limit_reached_trials", "patience_stopped_trials")})
print(audit["dataset_verification_scope"])


## 2. Toàn bộ trials và cấu hình được chọn

Trial giữ nguyên T0–T2; best epoch là 1-based. Những trial có hyperparameters khác nhau không phải các seed lặp để tính standard deviation hoặc confidence interval.


In [ ]:
columns = ["model", "trial", "selected", "learning_rate", "batch_size", "dropout", "mask_ratio", "soft_label_gamma", "lambda_0", "beta_0", "validation_ndcg@10", "best_epoch", "epochs_completed", "max_epochs", "stopping_reason", "training_seconds", "peak_gpu_memory_mib"]
display(trials[columns])
display(trials.loc[trials.selected, columns])


## 3. Convergence, stopping và tài nguyên

Chỉ vẽ validation checks thực sự; giá trị carry-forward không phải quan sát mới. Loss khác objective không so sánh trực tiếp giữa mô hình. Thời gian clustering đã nằm trong training time. Component logs không phải session wall time. Patience không phải chứng minh hội tụ toàn cục.


In [ ]:
from gen_fig_phase5c import generate_figures
figures = generate_figures(ROOT)
for path in figures:
    if path.suffix == ".png":
        display(Image(filename=str(path)))
print("Exported", len(figures), "files: PNG 300 DPI and vector PDF")


## 4. Báo cáo tiếng Việt và bước tiếp theo

Không thay đổi giao thức hoặc mở lại tuning cho trial chạm giới hạn. Final seeds, E1–E5, E7/E8 từ final checkpoints và TIGER/E6 vẫn là công việc tiếp theo. Training/testing/evaluation/đo GPU chạy trên Kaggle; phân tích/publication ở máy cá nhân.


In [ ]:
from write_phase5c_report import write_report
report_path = write_report(ROOT)
display(Markdown(report_path.read_text(encoding="utf-8")))
